# Task 03 - Decision Tree Classifier\n\nBuild a decision tree classifier using the Bank Marketing dataset to predict whether a client subscribes to a term deposit.

In [ ]:
from pathlib import Path\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder\nfrom sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/Prodigy-InfoTech/data-science-datasets/main/Task%203/bank/bank.csv"\ndf = pd.read_csv(DATA_URL, sep=";")\nprint("Dataset shape:", df.shape)\nprint(df.head())\nprint("\\nTarget distribution:")\nprint(df["y"].value_counts())

In [ ]:
X = df.drop(columns=["y"])\ny = df["y"].map({"no": 0, "yes": 1})\ncategorical_features = X.select_dtypes(include=["object"]).columns.tolist()\nnumeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()\n\npreprocessor = ColumnTransformer([\n    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),\n    ("numeric", "passthrough", numeric_features),\n])

In [ ]:
model = DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)\npipeline = Pipeline([("preprocessor", preprocessor), ("classifier", model)])\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.20, random_state=42, stratify=y\n)\npipeline.fit(X_train, y_train)\ny_pred = pipeline.predict(X_test)\n\nprint("Accuracy :", accuracy_score(y_test, y_pred))\nprint("Precision:", precision_score(y_test, y_pred, zero_division=0))\nprint("Recall   :", recall_score(y_test, y_pred, zero_division=0))\nprint("F1-score :", f1_score(y_test, y_pred, zero_division=0))\nprint(classification_report(y_test, y_pred, target_names=["No", "Yes"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)\nConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"]).plot()\nplt.title("Bank Marketing Decision Tree - Confusion Matrix")\nplt.tight_layout()\nplt.show()

In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()\nimportances = pipeline.named_steps["classifier"].feature_importances_\nimportance_df = (\n    pd.DataFrame({"feature": feature_names, "importance": importances})\n    .sort_values("importance", ascending=False)\n    .head(15)\n)\nplt.figure(figsize=(10, 7))\nplt.barh(importance_df["feature"].iloc[::-1], importance_df["importance"].iloc[::-1])\nplt.xlabel("Importance")\nplt.ylabel("Feature")\nplt.title("Top 15 Decision Tree Feature Importances")\nplt.tight_layout()\nplt.show()

In [ ]:
plt.figure(figsize=(22, 12))\nplot_tree(\n    pipeline.named_steps["classifier"],\n    feature_names=feature_names,\n    class_names=["No", "Yes"],\n    filled=True,\n    max_depth=3,\n    fontsize=7,\n)\nplt.title("Decision Tree Classifier (First 3 Levels)")\nplt.tight_layout()\nplt.show()